# 第 3 章 §3.3 —— Parallelization 逐段拆解

> 这是书里 `Chapter_03_Parallelization_(LangChain)` 的**注释版 + 本机适配版**。
> 【适配】原版用 `ChatOpenAI(model="gpt-4o-mini")`，本机换成 `common.make_llm()`
> （DeepSeek，OpenAI 兼容）。prompt、链结构、`RunnableParallel` 用法均未改动。

## 和前两章的关系

| | Ch1 Chaining | Ch2 Routing | **Ch3 Parallelization** |
|---|---|---|---|
| 控制流 | 直线 | 分岔（选一条） | **同时走多条，再汇合** |
| 模型调用数 | 2 次串行 | 1 次分类 + 1 次执行 | **N 次并发 + 1 次汇合** |
| 前提 | 步骤有依赖 | 需要分类器 | **分支必须互不依赖** |
| 收益来源 | 可靠性 | 适应性 | **省时间** |

一句话：**前三章的"原子操作"到此凑齐了**——串行、分支、并发。
后面所有复杂 agent 都是这三种的组合。

## 形状必须记住：扇出 → 汇合

```
                    ┌── summarize ──┐
   topic ── fan-out ├── questions ──┤ fan-in ── synthesis ── answer
                    └── key_terms ──┘
```

- **扇出（fan-out）**：三条链同时启动，各自独立
- **汇合（fan-in）**：synthesis **必须等三路全部完成**——这一步注定串行

## 本机实测

| 方式 | 加速比 |
|---|---|
| 串行 3 次调用 | 1.00x（基准） |
| `RunnableParallel` | **2.8 ~ 2.9x** |

理论极限 3x。贴近极限说明瓶颈在**等网络**，不是算力。
这一点第 3 节会展开（书里那句 GIL 提醒）。

想动手调试用 `../../ch03_parallelization/03_3_debug_lab.ipynb`（25 格，含 6 个实验）。

In [1]:
# =============================================================================
# 第 3 章 §3.3 完整代码（逐行注释 + 本机适配）
# =============================================================================
#
# 和 Ch1/Ch2 一样分两个阶段：
#   阶段 A（①—④）：搭积木，一张网络请求都不发
#   阶段 B（⑤）  ：invoke/ainvoke 点火，三条链同时起飞
#
# 唯一的新东西是 RunnableParallel：它把"纵向的管道"变成"横向的扇出"。

import sys
import asyncio
import threading
from pathlib import Path

# 【适配】向上找到含 common/ 的目录，才能 import make_llm
_root = Path.cwd().resolve()
for _d in (_root, *_root.parents):
    if (_d / "common").is_dir():
        sys.path.insert(0, str(_d))
        break

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

from common import make_llm


# ─────────────────────────────────────────────────────────────────────────────
# ① 模型客户端
# ─────────────────────────────────────────────────────────────────────────────
# 【适配】原版：ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
# 注意这里**只有一个 llm 对象**，却被三条链共用。
# 对象复用没问题：每次 .invoke() 都是一次独立的 HTTP 请求。
llm = make_llm(temperature=0.7)


# ─────────────────────────────────────────────────────────────────────────────
# ② 三条【互不依赖】的链 —— 这是能并行的唯一前提
# ─────────────────────────────────────────────────────────────────────────────
# 判断标准只看输入：三条链的输入都是同一个 {topic}，
# 谁也不吃谁的输出 → 可以并行。
# 如果改成 "把 summary 喂给 questions"，就必须串行（回到第 1 章）。

def make_chain(system_message: str):
    """同一个模板，只是 system 指令不同 —— 三条链的唯一差别就在这里。"""
    return (
        ChatPromptTemplate.from_messages([
            ("system", system_message),
            ("user", "{topic}"),
        ])
        | llm
        | StrOutputParser()
    )


summarize_chain = make_chain("Summarize the following topic concisely:")
questions_chain = make_chain("Generate three interesting questions about the following topic:")
terms_chain     = make_chain("Identify 5-10 key terms from the following topic, separated by commas:")


# ─────────────────────────────────────────────────────────────────────────────
# ③ RunnableParallel：扇出
# ─────────────────────────────────────────────────────────────────────────────
# 字典的键名 = 输出字段名。三条链会**同时**收到同一份输入。
#
# ★ 为什么必须有 "topic": RunnablePassthrough() ？
#   RunnableParallel 的输出【只包含它自己算出来的键】——
#   原始输入默认会被丢掉。而下一步的 synthesis_prompt 需要 {topic}，
#   少了这一行就会 KeyError: 'topic'。
#
#   记忆法：Passthrough = "把原始输入原样捎带过去"。
#   第 2 章也用过它（{"decision": ..., "request": RunnablePassthrough()}），
#   同一个工具、同一个用途。
map_chain = RunnableParallel(
    summary=summarize_chain,
    questions=questions_chain,
    key_terms=terms_chain,
    topic=RunnablePassthrough(),
)


# ─────────────────────────────────────────────────────────────────────────────
# ④ 汇合点：合成为最终答案
# ─────────────────────────────────────────────────────────────────────────────
# 这一步【必然串行】：它要等三路都回来。
# 也就是说整条链的耗时 ≈ max(三条链) + synthesis，而不是三者相加。
synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", """Based on the following information:
    Summary: {summary}
    Related Questions: {questions}
    Key Terms: {key_terms}
    Synthesize a comprehensive answer."""),
    ("user", "Original topic: {topic}"),
])

full_parallel_chain = map_chain | synthesis_prompt | llm | StrOutputParser()


# ─────────────────────────────────────────────────────────────────────────────
# ⑤ 运行
# ─────────────────────────────────────────────────────────────────────────────
async def run_parallel_example(topic: str) -> None:
    """书里的原函数。注意它用的是 ainvoke（异步）。"""
    print(f"\n--- Running Parallel LangChain Example for Topic: '{topic}' ---")
    response = await full_parallel_chain.ainvoke({"topic": topic})
    print("\n--- Final Response ---")
    print(response)


def run_in_own_loop(async_fn):
    """【适配】在独立线程 + 独立事件循环里跑协程。

    书里直接写 `asyncio.run(...)`，那在 .py 脚本里没问题；
    但 notebook 本身已经在一个事件循环里，asyncio.run() 会报
    "cannot be called from a running event loop"。

    ⚠️ 为什么不用 nest_asyncio：那要额外装包。
    ⚠️ 为什么不在循环里逐个 asyncio.run：httpx 的异步客户端会绑定在
       第一个事件循环上，换循环就报 "Event loop is closed"。
       所以这里传入的是"一次性跑完所有异步调用"的函数。
    """
    box = {}

    def _target():
        try:
            box["result"] = asyncio.run(async_fn())
        except BaseException as e:            # noqa: BLE001
            box["error"] = e

    th = threading.Thread(target=_target)
    th.start()
    th.join()
    if "error" in box:
        raise box["error"]
    return box["result"]


# ─────────────────────────────────────────────────────────────────────────────
# ⑥ 点火
# ─────────────────────────────────────────────────────────────────────────────
# 书里是 `if __name__ == "__main__": asyncio.run(...)`。
# 在 notebook 里 __name__ 也是 "__main__"，但事件循环已在运行，
# 所以这里显式调用我们的线程包装。
if __name__ == "__main__":
    run_in_own_loop(lambda: run_parallel_example("The history of space exploration"))


--- Running Parallel LangChain Example for Topic: 'The history of space exploration' ---



--- Final Response ---
## The history of space exploration: a synthesis

Space exploration grew out of early rocket theory and experiment—Tsiolkovsky’s mathematical work, Goddard’s liquid-fuel tests, and Germany’s V-2—then accelerated sharply during the Cold War. The Soviet Union’s Sputnik in 1957 and Yuri Gagarin’s orbital flight in 1961 turned space into a arena of superpower prestige. The United States responded with Mercury, Gemini, and Apollo, culminating in Apollo 11’s Moon landing in 1969. Later decades diversified the story: space stations (Salyut, Skylab, Mir, the International Space Station), robotic probes across the solar system, Mars rovers, the Space Shuttle, and satellite communications and navigation. Since the 2000s, private firms such as SpaceX have lowered launch costs and expanded commercial spaceflight, while current ambitions focus on returning to the Moon, exploring Mars, and pushing deeper into the solar system.

### 1. How much did Cold War rivalry accelerate 

## 执行时序：`invoke()` 内部到底发生了什么

| # | 环节 | 类型 | 花 token | 说明 |
|---|---|---|---|---|
| 1 | 输入 `{"topic": ...}` | — | ❌ | 一份输入，复制给三个分支 |
| 2 | `summarize_chain` | **网络** | ✅ | ┐ |
| 3 | `questions_chain` | **网络** | ✅ | ├ **三路并发** |
| 4 | `terms_chain` | **网络** | ✅ | ┘ |
| 5 | `RunnablePassthrough` | 本地 | ❌ | 原始 topic 原样带过 |
| 6 | 汇总成 dict | 本地 | ❌ | 4 个键：summary/questions/key_terms/topic |
| 7 | `synthesis_prompt` 插值 | 本地 | ❌ | 把 4 个值填进模板 |
| 8 | `llm` 汇合调用 | **网络** | ✅ | 串行，必须等前面全部完成 |
| 9 | `StrOutputParser` | 本地 | ❌ | → 最终字符串 |

**关键数字**：

- **模型调用 4 次**，其中前 3 次并发
- 总耗时 ≈ `max(3 条链) + synthesis`，不是 `3 条相加 + synthesis`
- 本机实测：串行 6~10s → 并行 2.3~3.5s，**加速比 2.8~2.9x**（理论上限 3x）

**关键细节**：第 3 步的"并发"发生在**同一个线程**里。
`RunnableParallel` 内部用线程池（`invoke`）或事件循环（`ainvoke`）来切换任务，
因为每条链绝大部分时间在**等网络响应**，切换才有价值。

## 七个关键点

### ① 并行只对 I/O 密集有效

书里这段提醒非常重要（原文第 7 页）：

> "Note that asyncio provides **concurrency, not parallelism**. It achieves this on a
> single thread by using an event loop ... constrained by Python's Global Interpreter
> Lock (GIL)."

**并发（concurrency）≠ 并行（parallelism）。**
一个线程在多个任务之间切换，只是因为它们在等待。
如果三条链是纯 CPU 计算，GIL 会让它们排队，加速比接近 1.0，甚至更慢。

> 判断口诀：**收益来自"等待"，不是来自"计算"。**

### ② `RunnablePassthrough()` 不是可选项

`RunnableParallel` 的输出只含它算出来的键。原始输入默认被丢弃。
下游若要 `{topic}`，就必须显式透传——否则 `KeyError: 'topic'`。

### ③ `invoke` 和 `ainvoke` 都能并行

书里用 async 容易让人以为"异步才有并行"，其实不是。
`invoke` 内部用线程池 executor，一样并发。实测两者耗时量级相同。

**用 async 的真正理由**：不阻塞调用方。在 Web 服务里，
阻塞式 `invoke` 会占住整个事件循环，`ainvoke` 才能同时服务其他请求。

### ④ 并行的失败是 all-or-nothing

任一条分支抛异常，整条 `RunnableParallel` 就抛异常，
**其他分支已经花掉的 token 白费**。

上生产时应该**在每个分支内部**加 `try/except`，让异常变成数据，
而不是让整条链崩掉。

### ⑤ 瞬时并发 = 撞限流

并行把 3 次调用压成一个瞬间。下游若有 RPM 限制，串行"慢慢跑"没事，
并行"一波打满"直接 429。

> 本学习项目真实踩过：Google 免费层 5 RPM，ADK 一个请求就要 2~3 次调用，
> 串行都不够用。**并行的第一成本不是 CPU，是配额。**

### ⑥ 汇合步骤注定串行

扇出再快，`synthesis` 也必须等三路齐全。所以：

```
总耗时 ≈ 最慢的那条分支 + 汇合生成
```

如果有一条分支特别慢，加速比会迅速趋近 1。**并行不等于均衡。**

### ⑦ 并行牺牲了"渐进式响应"

串行时用户能更早看到第一段结果；并行必须等最慢的那条才能继续。
所以对"流式输出优先"的场景，并行反而是倒退。

## 书里没给但很关键：LangGraph 的并行拓扑

书里说（§3.2）：

> "LangGraph ... allows you to define multiple nodes that can be executed from a
> single state transition, effectively enabling parallel branches."

这是对的，但**没给代码**（和 Ch2 的情况一样）。等价写法是：

```python
g.add_edge(START, "summarize")   # 三个节点都从 START 出发 = 扇出
g.add_edge(START, "questions")
g.add_edge(START, "key_terms")
for n in ("summarize", "questions", "key_terms"):
    g.add_edge(n, "synthesize")  # 三个都连汇合点 = 扇入（等全部完成）
```

完整可跑版本见 `../../ch03_parallelization/03_3_debug_lab.ipynb` 第 8 节。

**和 `RunnableParallel` 的差别**：LangGraph 的并行由 **state** 驱动，
每条分支写 state 的不同键，汇合节点天然能看到全部；
好处是中间态显式可见、分支可单独重试、后续能接循环。

## 坑清单

| # | 坑 | 现象 | 处置 |
|---|---|---|---|
| 1 | 忘了 `RunnablePassthrough()` | `KeyError: 'topic'` | 显式透传 |
| 2 | 分支之间有隐藏依赖 | 结果错乱或报错 | 检查输入，有依赖就串行 |
| 3 | CPU 密集也上并行 | 没加速甚至更慢 | 记住 GIL |
| 4 | 一条分支抛异常 | 整体失败，token 白费 | 分支内 try/except 降级 |
| 5 | 瞬时并发撞 RPM | 429 | 算配额 / 加信号量限流 |
| 6 | 分支耗时悬殊 | 加速比趋近 1 | 拆分或异步化长任务 |
| 7 | notebook 里 `asyncio.run` | "cannot be called from a running event loop" | 线程内跑，或顶层 await |

## 一句话总结

> **并行不是"更快的串行"，而是"不同的拓扑"。**
> 先问两件事：这些任务之间有没有依赖？它们是不是主要在等 I/O？
> 两个都是"是"，并行才有意义。